# PuckAPI: a season of hockey in five minutes

This notebook loads the free PuckAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: PuckAPI (puckapi.com), under the [PuckAPI Sample Data Licence](https://github.com/PuckAPI/mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/PuckAPI/mcp/releases/download/season-pack/puckapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('nhl', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## The final standings

In [ ]:
s = t('nhl', 'standings')
s.sort_values('points', ascending=False)[['team', 'points', 'goal_diff', 'xg_for', 'xg_against']].head(10)

## Goal difference against expected-goals difference (5v5)

In [ ]:
s['xg_diff'] = s.xg_for - s.xg_against
ax = s.plot.scatter(x='xg_diff', y='goal_diff', figsize=(7, 6), title='Goal difference vs xG difference')
for _, r in s.iterrows():
    ax.annotate(r['team'], (r['xg_diff'], r['goal_diff']), fontsize=7)

## Goalies by goals saved above expected, minimum 25 games

In [ ]:
g = t('nhl', 'goalies')
g[g.games_played >= 25].sort_values('gsax', ascending=False)[['goalie', 'team', 'games_played', 'save_pct', 'gsax', 'high_danger_save_pct']].head(10)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the PuckAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=puckapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/nhl/get_standings', data=b'{"division": "Atlantic"}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the hockey skills from https://github.com/PuckAPI/mcp.